# 🚀 Step 2 — 在 Kaggle GPU 上跑 Qwen3.8-27B 去审查版 LLM 服务器

这个 notebook 在 Kaggle 免费 GPU 上起一个**完整的 OpenAI 兼容 LLM 服务**，用 Cloudflare 隧道捅到公网 —— 不花钱、不要本地显卡。

### 架构

```
┌─────────────────────────────────────────────────────┐
│                  Kaggle GPU Server                  │
│                                                     │
│   llama-server (Qwen3.8-27B + mmproj)              │
│        ↕  localhost:8080                            │
│   cloudflared tunnel                                │
│        ↕  https://xxxx.trycloudflare.com            │
└─────────────────────────────────────────────────────┘
               ↕  internet
┌─────────────────────────────────────────────────────┐
│  你本机（任何 OpenAI 兼容客户端）                     │
│  → chat.py, Open WebUI, Chatbox, Python SDK...     │
└─────────────────────────────────────────────────────┘
```

### 为什么是这套

| 工具 | 作用 | 为什么不用另一个 |
|---|---|---|
| `llama.cpp` 原生二进制 | 推理 | CUDA 全权控制、`--mmproj` 多模态、没有 Python 开销 |
| `Cloudflare Tunnel` | 公网 URL | Ngrok 有请求数限制，还会掐断长 token 流 |
| `Kaggle` | 免费 GPU | Colab 约 1.5 小时掉线，Kaggle 能挂 12 小时 |

---

### ⚠️ 跑之前逐项确认

- [ ] **Accelerator** → **GPU T4 x2**（单卡能跑但慢）
- [ ] **Internet** → **ON**
- [ ] **Dataset** → `llama-cpp-bin` 挂在 *Input*（**只有这一个**）
- [ ] 模型不用建 Dataset —— `USE_HF_DIRECT = True` 直接从 HuggingFace 拉，约 2 分钟

> ⚠️ 想改成挂 Dataset：把 `USE_HF_DIRECT` 设成 `False`，填 `KAGGLE_USERNAME`，
> 再把 `qwen38-27b-abliterated-gguf` 也挂上。前提是 `1_setup_dataset.ipynb` 里的
> `kaggle datasets create` 真的成功了（那步需要 kaggle CLI 认证）。

> 显存账：Q5_K_XL 19.25 GB + mmproj 0.87 GB = 20.1 GB 权重，2×T4 池子 30 GB，
> 32k 上下文的 KV 只吃 2 GB（64 层里只有 16 层是 full attention，每 token 64 KiB），
> 剩下 7 GB 富余。

## ⚙️ Cell 1 — Configuration

**All parameters are here.** You should not need to change anything else in this notebook.

> 💡 **Thinking mode is not set here.** Control it per-conversation from your terminal:
> ```bash
> python chat.py --url https://xxxx.trycloudflare.com              # direct answers
> python chat.py --url https://xxxx.trycloudflare.com --thinking   # step-by-step reasoning
> ```
> No need to restart the server — switching modes is instant.

In [ ]:
import os
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — 只改这一格
# ══════════════════════════════════════════════════════════════════════════════

# ── 模型从哪来 ────────────────────────────────────────────────────────────────
# True  : 直接从 HuggingFace 拉（实测 21.6 GB / 1 分 45 秒 @540MB/s）
#         → 不需要建 Kaggle Dataset，也就不需要 kaggle CLI 认证
# False : 用你自己建的 Kaggle Dataset（要先跑 1_setup_dataset.ipynb 并成功上传）
USE_HF_DIRECT = True

HF_REPO     = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF"
HF_QUANT    = "UD-Q5_K_XL"                       # 19.25 GiB

MODEL_FILE  = "Huihui-Qwen3.8-27B-abliterated-UD-Q5_K_XL.gguf"
MMPROJ_FILE = "mmproj-model-bf16.gguf"           # 0.87 GiB

LOCAL_MODEL_DIR = Path("/tmp/model")             # /tmp 无 20GB 配额限制

# ── 只给 USE_HF_DIRECT = False 用 ─────────────────────────────────────────────
KAGGLE_USERNAME = ""                             # ← 用 Dataset 模式时填死你的用户名
DATASET_NAME    = "qwen38-27b-abliterated-gguf"

# ── llama.cpp 二进制（这个 Dataset 必须挂到 Input）────────────────────────────
LLAMA_DATASET          = "llama-cpp-bin"
LLAMA_BIN_DIR_OVERRIDE = ""                      # 留空自动在 /kaggle/input 里找含 llama-server 的目录

# ── 服务设置 ─────────────────────────────────────────────────────────────────
PORT         = 8080
MODEL_ALIAS  = "qwen3.8-27b-abliterated"
CTX_SIZE     = 32768   # 原生支持 262144；64k 要吃 4 GB KV，128k 吃 8 GB，别贪
EXTRA_FLAGS  = ["--jinja"]   # 让 llama-server 用 GGUF 里自带 chat template（thinking 开关靠它）

# ── 采样参数 ─────────────────────────────────────────────────────────────────
# Qwen3.8 官方推荐：
#   思考档   temperature=1.0  top_p=0.95  top_k=20  min_p=0  presence=0.0
#   非思考档 temperature=0.7  top_p=0.80  top_k=20  min_p=0  presence=1.5
# 这里写的是「非思考档」当服务端默认；chat.py --thinking 会按请求覆盖成思考档。
TEMPERATURE      = 0.7
TOP_P            = 0.8
TOP_K            = 20
MIN_P            = 0.0
PRESENCE_PENALTY = 1.5

# ── GPU 卸载 ─────────────────────────────────────────────────────────────────
# 999 = 能塞多少层塞多少层（自动停在显存上限）
# 1x T4 (15 GB): 部分层上卡 + 溢出到内存 → 能跑但慢
# 2x T4 (30 GB): 整模型进显存 → 全速
N_GPU_LAYERS = 999

# ══════════════════════════════════════════════════════════════════════════════

WORKING      = Path("/kaggle/working")
LLAMA_SERVER = WORKING / "llama-server"

# ── llama.cpp 二进制目录：扫 /kaggle/input 找 llama-server 本体 ───────────────
if LLAMA_BIN_DIR_OVERRIDE:
    LLAMA_BIN_DIR = Path(LLAMA_BIN_DIR_OVERRIDE)
else:
    hits = sorted({p.parent for p in Path("/kaggle/input").rglob("llama-server") if p.is_file()})
    if not hits:
        raise FileNotFoundError(
            "❌ /kaggle/input 里找不到 llama-server 二进制。\\n"
            "   把 llama-cpp-bin 这个 Dataset 挂到 Input（Add Input → Your Datasets）。\\n"
            "   如果还没建过，先跑 1b_setup_llama.ipynb（约 26 分钟编译一次，之后一直复用）。"
        )
    LLAMA_BIN_DIR = hits[0]
    if len(hits) > 1:
        print(f"⚠️  找到多个含 llama-server 的目录，用第一个：{LLAMA_BIN_DIR}")
        for h in hits[1:]:
            print(f"     忽略: {h}")

# ── 模型目录 ─────────────────────────────────────────────────────────────────
if USE_HF_DIRECT:
    MODEL_DIR = LOCAL_MODEL_DIR
else:
    if not KAGGLE_USERNAME:
        KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
    if not KAGGLE_USERNAME:
        raise ValueError("USE_HF_DIRECT=False 时必须填 KAGGLE_USERNAME。")
    MODEL_DIR = Path(f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{DATASET_NAME}")

MODEL_PATH  = MODEL_DIR / MODEL_FILE
MMPROJ_PATH = MODEL_DIR / MMPROJ_FILE

print("✅ Configuration loaded")
print(f"   模型来源     : {'HuggingFace 直连' if USE_HF_DIRECT else 'Kaggle Dataset'}")
print(f"   Model        : {MODEL_PATH}")
print(f"   mmproj       : {MMPROJ_PATH}")
print(f"   llama-cpp    : {LLAMA_BIN_DIR}")
print(f"   Alias        : {MODEL_ALIAS}")
print(f"   Context size : {CTX_SIZE} tokens")
if not MODEL_PATH.exists():
    print("   ⏳ 模型文件还没下载 —— 下一格会拉")


## ⬇️ Cell 2 — 准备模型文件

`USE_HF_DIRECT = True` 时直接从 HuggingFace 拉进 `/tmp/model`（无需 Kaggle Dataset、无需 CLI 认证）。
设成 `False` 就改读挂载的 Dataset。

> 校验点：两个文件必须都在，缺一个立刻报出目录里到底有什么。

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP: 准备模型文件
# ══════════════════════════════════════════════════════════════════════════════
import os, time

if USE_HF_DIRECT:
    from huggingface_hub import snapshot_download

    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"        # 新的高速后端
    os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)  # 已废弃，设了只会打 FutureWarning

    if MODEL_PATH.exists() and MMPROJ_PATH.exists():
        print("✅ 模型已在本地，跳过下载")
    else:
        LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
        print(f"⬇️  从 {HF_REPO} 拉 {HF_QUANT} + mmproj ...")
        print("   实测约 2 分钟（21.6 GB @540MB/s），比建 Dataset 快得多\n")
        t0 = time.time()
        snapshot_download(
            repo_id=HF_REPO,
            local_dir=str(LOCAL_MODEL_DIR),
            allow_patterns=[f"*{HF_QUANT}*", "*mmproj*"],
        )
        print(f"\n✅ 下载完成，耗时 {time.time() - t0:.0f} 秒")

    # 校验：这两个文件必须在，缺了直接炸，不让它带病往下走
    for f in (MODEL_PATH, MMPROJ_PATH):
        if not f.exists():
            listing = sorted(x.name for x in LOCAL_MODEL_DIR.iterdir()) if LOCAL_MODEL_DIR.exists() else []
            raise FileNotFoundError(
                f"❌ 缺文件：{f}\n"
                f"   {LOCAL_MODEL_DIR} 现有内容：{listing}\n"
                f"   → 检查 MODEL_FILE / MMPROJ_FILE 是否与仓库里的真实文件名一致。"
            )
else:
    print(f"使用 Kaggle Dataset 模式，直接从挂载点读：{MODEL_DIR}")
    for f in (MODEL_PATH, MMPROJ_PATH):
        if not f.exists():
            raise FileNotFoundError(
                f"❌ 缺文件：{f}\n"
                f"   → 确认 {DATASET_NAME} 已挂到 Input，且文件名对得上。"
            )

total = MODEL_PATH.stat().st_size + MMPROJ_PATH.stat().st_size
print(f"\n   Model  : {MODEL_PATH.name}  {MODEL_PATH.stat().st_size / 1024**3:.2f} GiB")
print(f"   mmproj : {MMPROJ_PATH.name}  {MMPROJ_PATH.stat().st_size / 1024**3:.2f} GiB")
print(f"   合计   : {total / 1024**3:.2f} GiB  （2x T4 显存池 30 GiB）")


## 🖥️ Cell 3 — GPU check

In [ ]:
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ No GPU detected. Go to Settings → Accelerator and select T4 GPU.")
else:
    lines = result.stdout.strip().split("\n")
    total_vram = 0
    print(f"{'GPU':<5} {'Name':<20} {'Total VRAM':>12} {'Free VRAM':>12}")
    print("-" * 55)
    for i, line in enumerate(lines):
        name, total, free = [x.strip() for x in line.split(",")]
        total_mb = int(total.replace(" MiB", ""))
        free_mb  = int(free.replace(" MiB", ""))
        total_vram += total_mb
        print(f"  {i:<3} {name:<20} {total_mb/1024:>10.1f} GB {free_mb/1024:>10.1f} GB")
    print("-" * 55)
    print(f"  Total available VRAM: {total_vram/1024:.1f} GB")
    print()
    model_size_gb = MODEL_PATH.stat().st_size / (1024**3)
    if total_vram / 1024 >= model_size_gb:
        print(f"✅ Enough VRAM ({total_vram/1024:.1f} GB) for the full model ({model_size_gb:.1f} GB) on GPU.")
    else:
        print(f"⚠️  VRAM ({total_vram/1024:.1f} GB) < model size ({model_size_gb:.1f} GB).")
        print(f"   Hybrid mode: GPU layers + RAM overflow. Works, but slower.")
        print(f"   → For full speed: switch to 2x T4 in Kaggle accelerator settings.")

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP: Install CUDA Runtime Dependencies
# Fix missing libcudart.so.12 / libcublas.so.12
# FIX BY APPSTOY.COM
# ══════════════════════════════════════════════════════════════════════════════

import os
import sys
import site
import subprocess
from pathlib import Path

print("Installing CUDA 12 runtime libraries...")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "nvidia-cuda-runtime-cu12",
    "nvidia-cublas-cu12"
], check=True)

# ── Locate NVIDIA shared libraries ────────────────────────────────────────────

cuda_lib_dirs = []

for base in site.getsitepackages():
    nvidia_dir = Path(base) / "nvidia"

    if nvidia_dir.exists():
        for lib_name in ["libcudart.so.12", "libcublas.so.12"]:
            for lib in nvidia_dir.rglob(lib_name):
                cuda_lib_dirs.append(str(lib.parent))

# Remove duplicate paths
cuda_lib_dirs = list(dict.fromkeys(cuda_lib_dirs))

# ── Configure dynamic library search path ────────────────────────────────────

old_path = os.environ.get("LD_LIBRARY_PATH", "")

os.environ["LD_LIBRARY_PATH"] = ":".join(
    cuda_lib_dirs + ["/kaggle/working", old_path]
)

print("\nCUDA library directories:")
for path in cuda_lib_dirs:
    print("  ", path)

# ── Verify required libraries ────────────────────────────────────────────────

for lib_name in ["libcudart.so.12", "libcublas.so.12"]:
    found = any(
        (Path(path) / lib_name).exists()
        for path in cuda_lib_dirs
    )
    print(f"{'✅' if found else '❌'} {lib_name}")

print("\n✅ CUDA Runtime setup completed.")


## 📦 Cell 4 — Load llama.cpp binaries from dataset

We copy the pre-compiled binaries **and** their shared libraries (`.so` files) from the
`llama-cpp-bin` dataset into `/kaggle/working/` (writable).

Then we set `LD_LIBRARY_PATH` so the OS finds the `.so` files at runtime.
Without this, `llama-server` crashes immediately with `cannot open shared object file`.

> ⏳ Expected time: **~5 seconds** (vs ~26 minutes of compilation).

In [ ]:
import shutil, json, os, subprocess
from pathlib import Path

# ── Copy binaries ─────────────────────────────────────────────────────────────
binaries = ["llama-server", "llama-cli", "llama-mtmd-cli"]
for name in binaries:
    src = LLAMA_BIN_DIR / name
    dst = WORKING / name
    assert src.exists(), (
        f"❌ Binary '{name}' not found in dataset.\n"
        f"   Re-run 1b_setup_llama.ipynb to rebuild the dataset."
    )
    shutil.copy2(src, dst)
    dst.chmod(0o755)

# ── Copy shared libraries (.so) ───────────────────────────────────────────────
# llama-server is dynamically linked — it needs its .so files at runtime.
# We copy them to /kaggle/working/ alongside the binaries.
so_count = 0
for so_file in LLAMA_BIN_DIR.glob("*.so*"):
    shutil.copy2(so_file, WORKING / so_file.name)
    so_count += 1

# ── Set LD_LIBRARY_PATH ───────────────────────────────────────────────────────
# Tell the OS where to find the .so files when llama-server starts.
ld_path = f"/kaggle/working:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = ld_path

# ── Verify binary works ───────────────────────────────────────────────────────
test = subprocess.run(
    [str(LLAMA_SERVER), "--version"],
    capture_output=True, text=True,
    env={**os.environ, "LD_LIBRARY_PATH": ld_path}
)

if test.returncode != 0:
    print("❌ llama-server failed sanity check:")
    print(test.stderr[:500])
    raise RuntimeError("Binary verification failed — check dataset contents.")

# ── Print summary ─────────────────────────────────────────────────────────────
ver_file = LLAMA_BIN_DIR / "version.json"
if ver_file.exists():
    info = json.loads(ver_file.read_text())
    print("✅ llama.cpp binaries loaded.")
    print(f"   Commit  : {info.get('llama_cpp_commit')}  ({info.get('commit_date')})")
    print(f"   Version : {info.get('llama_server_ver')}")
    print(f"   CUDA    : {info.get('cuda_arch')}")
else:
    ver_str = (test.stdout or test.stderr).strip().splitlines()[0]
    print(f"✅ llama.cpp binaries loaded: {ver_str}")

print(f"   Copied  : {len(binaries)} binaries + {so_count} shared libraries")
print(f"   LD path : {ld_path.split(':')[0]}")

## ☁️ Cell 5 — Install Cloudflare Tunnel

`cloudflared` creates a secure public URL forwarding traffic to our local server.

Why Cloudflare over Ngrok?
- **No account or token needed** — works instantly in anonymous mode
- **No request limits** — Ngrok's free tier blocks the API after too many calls
- **No streaming timeout** — Ngrok cuts long HTTP connections, breaking LLM token streaming

In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
     -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
echo "✅ cloudflared installed: $(cloudflared --version)"

## 🧠 Cell 6 — Launch llama-server

Starts `llama-server` in the background, then waits until `/health` returns `{"status": "ok"}`
before continuing. Checking for `ok` (not just HTTP 200) ensures the model is fully loaded.

**Thinking mode** is not set here — it is passed per request by `chat.py`.
No server restart needed to switch modes.

In [ ]:
import subprocess, time, requests, os

# Kill any leftover llama-server from a previous run
subprocess.run(["pkill", "-f", "llama-server"], capture_output=True)
time.sleep(2)

# LD_LIBRARY_PATH must be in the environment of the subprocess too
env = {**os.environ, "LD_LIBRARY_PATH": f"/kaggle/working:{os.environ.get('LD_LIBRARY_PATH', '')}"}

cmd = [
    str(LLAMA_SERVER),
    "--model",            str(MODEL_PATH),
    "--mmproj",           str(MMPROJ_PATH),      # 视觉投影层 — 开图片输入
    "--alias",            MODEL_ALIAS,
    "--host",             "0.0.0.0",
    "--port",             str(PORT),
    "--n-gpu-layers",     str(N_GPU_LAYERS),
    "--ctx-size",         str(CTX_SIZE),
    "--temp",             str(TEMPERATURE),
    "--top-p",            str(TOP_P),
    "--top-k",            str(TOP_K),
    "--min-p",            str(MIN_P),
    "--presence-penalty", str(PRESENCE_PENALTY),
    "--log-file",         "/tmp/llama-server.log",
    *EXTRA_FLAGS,
    # Thinking 模式不在这里定死 —— chat.py 每次请求按需覆盖，不用重启
]

print("🚀 Starting llama-server...")
print(f"   Model  : {MODEL_FILE}")
print(f"   mmproj : {MMPROJ_FILE}")
print(f"   Flags  : {' '.join(EXTRA_FLAGS) if EXTRA_FLAGS else '(none)'}")
print()

server_proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env)

print("⏳ Loading model into GPU memory (19 GB from /kaggle/input, 3–6 minutes)...")

# Wait for {"status": "ok"} — not just HTTP 200
# llama-server returns 200 + {"status":"loading"} while loading,
# and 200 + {"status":"ok"} only when truly ready.
timeout, interval, elapsed, server_ok = 900, 15, 0, False
while elapsed < timeout:
    if server_proc.poll() is not None:
        print(f"\n❌ llama-server exited early with code {server_proc.returncode}")
        break
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=2)
        if r.status_code == 200 and r.json().get("status") == "ok":
            server_ok = True
            break
    except Exception:
        pass
    print(f"   [{elapsed:>3}s] Still loading...", flush=True)
    time.sleep(interval)
    elapsed += interval

if not server_ok:
    print("\n❌ Server failed to start. Last log lines:")
    try:
        with open("/tmp/llama-server.log") as f:
            print("".join(f.readlines()[-30:]))
        print("\n   ↑ 若是 'error: unknown argument' → EXTRA_FLAGS 里的开关这个 build 不认，删掉重跑。")
        print("   ↑ 若是 'error loading model' / 'missing tensor' → 换 UD-Q4_K_XL（16.19 GB）试。")
    except Exception:
        print("   (log file not found — binary may have crashed before writing logs)")
    raise RuntimeError("Server did not become healthy in time.")

print(f"\n✅ Server is ready! (started in {elapsed}s)")
print(f"   Local endpoint: http://localhost:{PORT}/v1")

# ── 确认 arch 是 qwen35（不是 qwen35moe）────────────────────────────────────
try:
    with open("/tmp/llama-server.log") as f:
        arch = [l.strip() for l in f if "architecture" in l.lower()][:3]
    if arch:
        print("\n   arch 检查:")
        for a in arch:
            print(f"     {a}")
        print("   ↑ 应该是 qwen35（dense 27B）。若是 qwen35moe 说明模型文件拿错了。")
except Exception:
    pass

## 🌐 Cell 7 — Start Cloudflare Tunnel & get public URL

In [ ]:
import re

tunnel_log = "/tmp/cloudflare.log"
tunnel_url = None

with open(tunnel_log, "w") as log:
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
        stdout=log, stderr=log,
    )

print("⏳ Waiting for Cloudflare tunnel URL...")

for _ in range(30):
    time.sleep(2)
    try:
        with open(tunnel_log) as f:
            match = re.search(r"https://[\w-]+\.trycloudflare\.com", f.read())
        if match:
            tunnel_url = match.group(0)
            break
    except Exception:
        pass

if not tunnel_url:
    print("❌ Could not get Cloudflare URL. Last log:")
    with open(tunnel_log) as f:
        print(f.read()[-1000:])
    raise RuntimeError("Cloudflare tunnel failed to start.")

print()
print("═" * 60)
print(f"  🔗 YOUR SERVER URL")
print(f"  {tunnel_url}")
print("═" * 60)
print()
print(f"  OpenAI endpoint : {tunnel_url}/v1")
print(f"  Direct answers  : python chat.py --url {tunnel_url}")
print(f"  Thinking mode   : python chat.py --url {tunnel_url} --thinking")

## 🔁 Cell 8 — Keep-alive loop

Keeps the notebook alive for up to 12 hours. Prints a health check every 5 minutes.

**To stop:** interrupt the kernel (■ button) or let the session expire.

In [ ]:
import datetime

start_time     = datetime.datetime.now()
check_interval = 300

print(f"🟢 Server running since {start_time.strftime('%H:%M:%S')}")
print(f"   URL     : {tunnel_url}")
print(f"   Model   : {MODEL_FILE}")
print(f"   Timeout : ~12h (Kaggle session limit)")
print()
print(f"   Connect : python chat.py --url {tunnel_url}")
print(f"   Thinking: python chat.py --url {tunnel_url} --thinking")
print()
print("   Press ■ (interrupt kernel) to stop.")
print("─" * 60)

try:
    while True:
        time.sleep(check_interval)
        elapsed = datetime.datetime.now() - start_time
        h, rem  = divmod(int(elapsed.total_seconds()), 3600)
        m       = rem // 60
        try:
            r      = requests.get(f"http://localhost:{PORT}/health", timeout=5)
            status = "✅ healthy" if r.status_code == 200 else f"⚠️ {r.status_code}"
        except Exception:
            status = "❌ unreachable"
        ts = datetime.datetime.now().strftime('%H:%M:%S')
        print(f"  [{ts}]  Uptime: {h}h {m:02d}m  |  {status}")
        if "❌" in status:
            print("\n⚠️  Server crashed. Check /tmp/llama-server.log")
            break
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")
    server_proc.terminate()
    tunnel_proc.terminate()
    print("   Done.")